In [0]:
%sql
use catalog training_dev;

In [0]:
landing_zone = '/Volumes/training_dev/default/raw'
order_file = landing_zone + '/order_hist'
checkpoint_location = landing_zone + '/order_checkpoint'

##### Setup #1:
- Create a **read stream** - reads the file at location and continuously scans for data
- Create a **write stream** - writes to managed table - execute when the table needs to be loaded
- keep dropping in incremental files and run the write stream and monitor the load status.
- change the datatype and schema in source and notice behaviour

In [0]:
order_read_df = (spark.readStream.format('cloudFiles')\
    .option("cloudFiles.format", "csv")\
    .option("cloudFiles.inferSchema", 'true')\
    .option("cloudFiles.inferColumnTypes", "true")\
    .option("cloudFiles.schemaLocation", checkpoint_location)\
    .load(order_file)
    )

In [0]:
%sql
drop table if exists training_dev.default.order_m1;

In [0]:
order_read_df.writeStream.format('delta') \
.option("checkpointLocation", checkpoint_location) \
.option("mergeSchema",'true') \
.outputMode('append') \
.trigger(availableNow = True) \
.toTable('training_dev.default.order_m3')


In [0]:
%sql
select * from training_dev.default.order_m;

- read stream keeps on running. 
- write stream when executed everytime loads the data from the read stream to managed table
- Incremental file at location - if schema perfectly matching - is loaded into target without failure - when writestream is executed.
- In event, schema evolves at source side, table write fails. Remedy - re-initiate the read stream and then trigger the writestream.
- When there is data type mismatch (source data type changes - here customer_id = string and NOT integer) then it DOESN'T fail, problematic column appears in the _reascued_path column.
- This processes incremental data only, using checkpoint_location
- Meaning, loading same file doesnt happen, clear the checkpoint location and then re-load the file.      

##### Setup #2:
- improve setup #1 to **enforce schema** rather than inferring
- add audit columns

In [0]:
%sql
drop table if exists training_dev.default.order_m;

In [0]:
ord_schema = 'order_id integer, order_date timestamp, customer_id integer, order_status string'

order_read_df = (spark.readStream.format('cloudFiles')\
    .option("cloudFiles.format", "csv")\
    .option("header","true") \
    .schema(ord_schema) \
    .option('rescuedDataColumn',"_rescued_data") \
    .option("cloudFiles.schemaLocation", checkpoint_location) \
    .load(order_file) \
    .withColumn("ingest_dt",current_timestamp()) \
    .withColumn("_file_nm",col('_metadata.file_path')) 
    )

In [0]:
from  pyspark.sql.functions import *

order_read_df.writeStream.format('delta') \
    .option("checkpointLocation", checkpoint_location) \
    .option("mergeSchema",'true') \
    .outputMode('append') \
    .trigger(availableNow = True) \
    .toTable('training_dev.default.order_m')

In [0]:
%sql
select * from training_dev.default.order_m;

#### Schema Evolution in Autoloader
- **none** (Default - when schema enforced) - no rescue, no failure - upon new column and datatype change. Rescue can be added explicitly.
- **failOnNewColumns** - schema mismatches leads to write failures, re-try doesnt fix. either change schema manually or ignore file.
- **rescue** - accepts all schema change, mismatches captured in _rescued_data field.
- **addNewColumns** (Default - when schema infer) - write fail on new column, re-try fixes it. data type mismatch doesnt fail writes.

In [0]:
from pyspark.sql.functions import *

order_read_df = (spark.readStream.format('cloudFiles')\
    .option("cloudFiles.format", "csv")\
    .option("cloudFiles.inferSchema", 'true')\
    .option("cloudFiles.inferColumnTypes", "true")\
    .option("cloudFiles.schemaLocation", checkpoint_location)\
    .option("cloudFiles.schemaEvolutionMode", "none")\
    .load(order_file)\
    .withColumn("ingest_dt",current_timestamp())
    .withColumn("_file_nm",col('_metadata.file_path'))
        )

In [0]:
from  pyspark.sql.functions import *

order_read_df.writeStream.format('delta') \
    .option("checkpointLocation", checkpoint_location) \
    .option("mergeSchema",'true') \
    .outputMode('append') \
    .trigger(availableNow = True) \
    .toTable('training_dev.default.order_m')

In [0]:
%sql

select * from  training_dev.default.order_m

In [0]:
%sql
drop table if exists  training_dev.default.order_m

In [0]:
from pyspark.sql.functions import *

order_read_df = (spark.readStream.format('cloudFiles')\
    .option("cloudFiles.format", "csv")\
    .option("cloudFiles.inferSchema", 'true')\
    .option("cloudFiles.inferColumnTypes", "true")\
    .option("cloudFiles.schemaLocation", checkpoint_location)\
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")\
    .load(order_file)\
    .withColumn("ingest_dt",current_timestamp())
    .withColumn("_file_nm",col('_metadata.file_path'))
        )

In [0]:
from  pyspark.sql.functions import *

order_read_df.writeStream.format('delta') \
    .option("checkpointLocation", checkpoint_location) \
    .option("mergeSchema",'true') \
    .outputMode('append') \
    .trigger(availableNow = True) \
    .toTable('training_dev.default.order_m')

    

In [0]:
%sql

select * from  training_dev.default.order_m

In [0]:
%sql
drop table if exists  training_dev.default.order_m

In [0]:
from pyspark.sql.functions import *

order_read_df = (spark.readStream.format('cloudFiles')\
    .option("cloudFiles.format", "csv")\
    .option("cloudFiles.inferSchema", 'true')\
    .option("cloudFiles.inferColumnTypes", "true")\
    .option("cloudFiles.schemaLocation", checkpoint_location)\
    .option("cloudFiles.schemaEvolutionMode", "failOnNewColumns")\
    .load(order_file)\
    .withColumn("ingest_dt",current_timestamp())
    .withColumn("_file_nm",col('_metadata.file_path'))
        )

In [0]:
from  pyspark.sql.functions import *

order_read_df.writeStream.format('delta') \
    .option("checkpointLocation", checkpoint_location) \
    .option("mergeSchema",'true') \
    .outputMode('append') \
    .trigger(availableNow = True) \
    .toTable('training_dev.default.order_m')

    

In [0]:
%sql

select * from  training_dev.default.order_m

In [0]:
%sql

drop table  training_dev.default.order_m

In [0]:
from pyspark.sql.functions import *

order_read_df = (spark.readStream.format('cloudFiles')\
    .option("cloudFiles.format", "csv")\
    .option("cloudFiles.inferSchema", 'true')\
    .option("cloudFiles.inferColumnTypes", "true")\
    .option("cloudFiles.schemaLocation", checkpoint_location)\
    .option("cloudFiles.schemaEvolutionMode", "rescue")\
    .load(order_file)\
    .withColumn("ingest_dt",current_timestamp())
    .withColumn("_file_nm",col('_metadata.file_path'))
        )

In [0]:
from  pyspark.sql.functions import *

order_read_df.writeStream.format('delta') \
    .option("checkpointLocation", checkpoint_location) \
    .option("mergeSchema",'true') \
    .outputMode('append') \
    .trigger(availableNow = True) \
    .toTable('training_dev.default.order_m')

    

In [0]:
%sql

select * from  training_dev.default.order_m

In [0]:
%sql
Drop table training_dev.default.order_m;

In [0]:
%sql
drop volume training_dev.default.raw;